# General Observing

This notebook is used for general observing with the Auxiliary Telescope.

Craig Lage - 25-May-21

In [1]:
import sys
import asyncio
import time
import os
import numpy as np

from lsst.ts import salobj
from lsst.ts.observatory.control.auxtel.atcs import ATCS
from lsst.ts.observatory.control.auxtel.latiss import LATISS
from lsst.ts.observatory.control.utils import RotType

In [2]:
# for tab completion to work in current notebook instance
%config IPCompleter.use_jedi = False

In [3]:
import logging
stream_handler = logging.StreamHandler(sys.stdout)
logger = logging.getLogger()
logger.addHandler(stream_handler)
logger.level = logging.DEBUG
# Make matplotlib less chatty
logging.getLogger("matplotlib").setLevel(logging.WARNING)

In [4]:
#Start classes
domain = salobj.Domain()
await asyncio.sleep(10) # This can be removed in the future...
atcs = ATCS(domain)
latiss = LATISS(domain)
await asyncio.gather(atcs.start_task, latiss.start_task)

atmcs: Adding all resources.
atptg: Adding all resources.
ataos: Adding all resources.
atpneumatics: Adding all resources.
athexapod: Adding all resources.
atdome: Adding all resources.
atdometrajectory: Adding all resources.
atcamera: Adding all resources.
atspectrograph: Adding all resources.
atheaderservice: Adding all resources.
atarchiver: Adding all resources.
Could not read historical data in 60.11 sec
Read 100 history items for RemoteEvent(ATHeaderService, 0, heartbeat)
Read 100 history items for RemoteEvent(ATHeaderService, 0, largeFileObjectAvailable)
Read 100 history items for RemoteEvent(ATHeaderService, 0, logMessage)
Read 4 history items for RemoteEvent(ATHeaderService, 0, summaryState)
Could not read historical data in 60.15 sec
Read 1 history items for RemoteEvent(ATDomeTrajectory, 0, appliedSettingsMatchStart)
Read 100 history items for RemoteEvent(ATDomeTrajectory, 0, heartbeat)
Read 3 history items for RemoteEvent(ATDomeTrajectory, 0, logMessage)
Read 6 history items

[[None, None, None, None, None, None, None], [None, None, None, None]]

trajectory DDS read queue is filling: 92 of 100 elements
torqueDemand DDS read queue is filling: 92 of 100 elements
nasymth_m3_mountMotorEncoders DDS read queue is filling: 92 of 100 elements
mount_Nasmyth_Encoders DDS read queue is filling: 92 of 100 elements
mount_AzEl_Encoders DDS read queue is filling: 92 of 100 elements
mount_AzEl_Encoders python read queue is filling: 91 of 100 elements
measuredTorque DDS read queue is filling: 93 of 100 elements
measuredMotorVelocity DDS read queue is filling: 93 of 100 elements
azEl_mountMotorEncoders DDS read queue is filling: 93 of 100 elements


In [ ]:
# enable components
await atcs.enable({"atdome": "current", "ataos": "current", "athexapod": "current"})
await latiss.enable({"atspectrograph": "current"})

In [ ]:
# If some components fail to enable, some set of commands like the ones below may be needed
# await salobj.set_summary_state(atcs.rem.atdome, salobj.State.ENABLED, settingsToApply='current')
# await latiss.rem.atarchiver.cmd_start.set_start(timeout=30)

In [ ]:
# Take a bias to make sure everything is working
await latiss.take_bias(1)

In [ ]:
# Take 50 biases seq # 006-055
# Added wait to stop killing the recent images
for i in range(50):
    await asyncio.sleep(2.0)
    await latiss.take_bias(1)

In [ ]:
# Take 10 10 second darks 56-65
await latiss.take_darks(10.0, 10)

In [14]:
atcs.check.atdome = True
atcs.check.atdometrajectory = True

In [8]:
await salobj.set_summary_state(atcs.rem.atdome, salobj.State.ENABLED, settingsToApply='current')

[<State.STANDBY: 5>, <State.DISABLED: 1>, <State.ENABLED: 2>]

In [ ]:
await salobj.set_summary_state(atcs.rem.atdometrajectory, salobj.State.ENABLED, settingsToApply='current')

In [ ]:
await atcs.prepare_for_flatfield()

In [ ]:
# Take a test flat
await latiss.take_flats(2.0, 1, filter='RG610', grating='empty_1')

In [ ]:
# Take 10 2 second flats 67-76
await latiss.take_flats(2.0, 10, filter='RG610', grating='empty_1')

In [ ]:
# Take flats for PTC 77-136
# Added wait to stop killing the recent images
for i in range(30):
    exp = 0.2 * float(i+1)
    await latiss.take_flats(exp, 1, filter='RG610', grating='empty_1')
    if exp < 2.0:
        await asyncio.sleep(2.0)
    await latiss.take_flats(exp, 1, filter='RG610', grating='empty_1')


In [9]:
# This moves everything to park position and opens the dome.  It takes 5+ minutes.
await atcs.prepare_for_onsky()

Enabling all components
Gathering settings.
atdometrajectory check is disabled, skipping.
Couldn't get settingVersions event. Using empty settings.
Couldn't get settingVersions event. Using empty settings.
Couldn't get settingVersions event. Using empty settings.
Complete settings for atmcs.
Complete settings for atptg.
Complete settings for ataos.
Complete settings for atpneumatics.
Complete settings for athexapod.
Complete settings for atdome.
Settings versions: {'atmcs': '', 'atptg': '', 'ataos': '', 'atpneumatics': '', 'athexapod': '', 'atdome': 'current'}
[atmcs]::[<State.ENABLED: 2>]
[atptg]::[<State.ENABLED: 2>]
[ataos]::[<State.ENABLED: 2>]
[atpneumatics]::[<State.ENABLED: 2>]
[athexapod]::[<State.ENABLED: 2>]
[atdome]::[<State.ENABLED: 2>]
[atdometrajectory]::<State.DISABLED: 1>
All components in <State.ENABLED: 2>.
Slew telescope to park position.
Sending command
Stop tracking.
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED

TimeoutError: 

In [11]:
await atcs.rem.atpneumatics.cmd_openM1CellVents.start()

In [ ]:
# This Opens/Closes the dropout shutter
# await atcs.rem.atdome.cmd_moveShutterDropoutDoor.set_start(open=False)

In [16]:
# This tunrs on ATAOS corrections and turns on the air pressure under the M1 mirror.
await atcs.rem.ataos.cmd_enableCorrection.set_start(
    m1=True, hexapod=True, atspectrograph=True, timeout=atcs.long_timeout
)

In [15]:
await atcs.enable({"atdome": "current", "ataos": "current", "athexapod": "current"})

Enabling all components
Gathering settings.
Received settings from users.: {'atdome': 'current', 'ataos': 'current', 'athexapod': 'current'}
Couldn't get settingVersions event. Using empty settings.
Couldn't get settingVersions event. Using empty settings.
Complete settings for atmcs.
Complete settings for atptg.
Complete settings for atpneumatics.
Complete settings for atdometrajectory.
Settings versions: {'atdome': 'current', 'ataos': 'current', 'athexapod': 'current', 'atmcs': '', 'atptg': '', 'atpneumatics': '', 'atdometrajectory': ''}
[atmcs]::[<State.ENABLED: 2>]
[atptg]::[<State.ENABLED: 2>]
[ataos]::[<State.ENABLED: 2>]
[atpneumatics]::[<State.ENABLED: 2>]
[athexapod]::[<State.ENABLED: 2>]
[atdome]::[<State.ENABLED: 2>]
[atdometrajectory]::[<State.DISABLED: 1>, <State.ENABLED: 2>]
All components in <State.ENABLED: 2>.


In [19]:
await atcs.rem.atdometrajectory.cmd_setFollowingMode.set_start(enable=True)

In [ ]:
# Pointing to a given Az/El  This will not track
await atcs.point_azel(az, el, rot_tel=rot)

In [ ]:
# Pointing to a given RA/Dec  This will track
await atcs.slew_icrs(ra=ra, dec=dec, rot=rot, rot_type=RotType.PhysicalSky)

In [17]:
# Slew to a given object and start tracking.
await atcs.slew_object('HD 94340', rot_type=RotType.Parallactic)

Starting new HTTP connection (1): simbad.u-strasbg.fr:80
http://simbad.u-strasbg.fr:80 "POST /simbad/sim-script HTTP/1.1" 200 None
Slewing to HD 94340: 10 53 04.5347 -20 37 41.542
Setting rotator position with respect to parallactic angle to 0.0 deg.
Parallactic angle: -138.25559310675953 | Sky Angle: -48.25559310675952
Sending command
Stop tracking.
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
atdome: <State.ENABLED: 2>
atdometrajectory: <State.ENABLED: 2>
[Telescope] delta Alt = -003.305 deg; delta Az = +047.183 deg; delta N1 = +000.000 deg; delta N2 = -014.310 deg [Dome] delta Az = -003.010 deg
ATDome in position.
[Telescope] delta Alt = -002.906 deg; delta Az = +045.353 deg; delta N1 = +000.000 deg; delta N2 = -014.108 deg [Dome] delta Az = -003.010 deg
[Telescope] delta Alt = -000.015 deg; delta Az = +039.329 deg; delta N1 = +000.000 d

In [18]:
# It is recommended to do this and wait several seconds for the spectrograph to settle
# before taking an image
await latiss.setup_atspec(filter='RG610', grating='empty_1')

In [20]:
# Take 1 2 second image
await latiss.take_object(2.0, 1, filter='RG610', grating='empty_1')

Generating group_id
imagetype: OBJECT, TCS synchronization not configured.
OBJECT 0001 - 0001


array([2021052500137])

In [29]:
await atcs.rem.ataos.cmd_disableCorrection.set_start(atspectrograph=True)

In [34]:
# This loop will take a set of images on both sides of current best focus
# starts at starting_z_offset and runs to -starting_z_offset
# then puts the z_offset back where it was.

starting_z_offset = -0.20
z_offset_increment = 0.05
nsteps = int((-2 * starting_z_offset) / z_offset_increment) + 1
total_z_offset = 0.0
await atcs.rem.ataos.cmd_offset.set_start(z=starting_z_offset)
total_z_offset += starting_z_offset
print(f"Total z offset = {total_z_offset}")
await asyncio.sleep(2)
for i in range(nsteps):
    await latiss.take_object(5.0, 1, filter='RG610', grating='holo4_003')
    await atcs.rem.ataos.cmd_offset.set_start(z=z_offset_increment)
    total_z_offset += z_offset_increment
    print(f"Total z offset = {total_z_offset}")
    
# Put offset back where it was
await atcs.rem.ataos.cmd_offset.set_start(z=-total_z_offset)
total_z_offset -= total_z_offset
print(f"Total z offset = {total_z_offset}")
    

Total z offset = -0.2
Generating group_id
imagetype: OBJECT, TCS synchronization not configured.
OBJECT 0001 - 0001
Total z offset = -0.15000000000000002
Generating group_id
imagetype: OBJECT, TCS synchronization not configured.
OBJECT 0001 - 0001
Total z offset = -0.10000000000000002
Generating group_id
imagetype: OBJECT, TCS synchronization not configured.
OBJECT 0001 - 0001
Total z offset = -0.05000000000000002
Generating group_id
imagetype: OBJECT, TCS synchronization not configured.
OBJECT 0001 - 0001
Total z offset = -1.3877787807814457e-17
Generating group_id
imagetype: OBJECT, TCS synchronization not configured.
OBJECT 0001 - 0001
Total z offset = 0.04999999999999999
Generating group_id
imagetype: OBJECT, TCS synchronization not configured.
OBJECT 0001 - 0001
Total z offset = 0.09999999999999999
Generating group_id
imagetype: OBJECT, TCS synchronization not configured.
OBJECT 0001 - 0001
Total z offset = 0.15
Generating group_id
imagetype: OBJECT, TCS synchronization not config

In [ ]:
# This will put in a hexapod offset
#await atcs.rem.ataos.cmd_offset.set_start(z=-0.095)

In [ ]:
# To reset all hexapod offsets
tmp = await atcs.rem.ataos.cmd_resetOffset.set_start(axis='y')

In [31]:
# Move the star within the field
# Offsets are in arcseconds.
await atcs.offset_xy(y=100, x=0, relative=True)

Calculating x/y offset: 0/100 
Applying Az/El offset: -73.33440514535982/-67.98577073164812 
Telescope not in position.
All axes in position.
Waiting for telescope to settle.
Done


In [32]:
await latiss.take_object(5.0, 1, filter='RG610', grating='ronchi170lpmm')

Generating group_id
imagetype: OBJECT, TCS synchronization not configured.
OBJECT 0001 - 0001


array([2021052500224])

In [22]:
atcs.offset_azel?

Signature: atcs.offset_azel(az, el, relative=True, persistent=None, absorb=False)
Docstring:
Offset telescope in azimuth and elevation.

For more information see the Notes section below or the package
documentation in https://ts-observatory-control.lsst.io/.

Parameters
----------
az : `float`
    Offset in azimuth (arcsec).
el : `float`
    Offset in elevation (arcsec).
relative : `bool`, optional
    If `True` (default) offset is applied relative to the current
    position, if `False` offset replaces any existing offsets.
persistent : `bool` or `None`, optional (deprecated)
    (Deprecated) Should the offset be absorbed and persisted between
    slews? Use of this parameter is deprecated. Use `absorb` instead.
absorb : `bool`, optional
    Should the offset be absorbed and persisted between slews?
    (default: `False`)

See Also
--------
offset_xy : Offsets in the detector X/Y plane.
offset_radec : Offset in sky coordinates.
reset_offsets : Reset offsets.

Notes
-----

The `persist

In [28]:
# Run a figure-8 of offsets
offsets = [[0,0], [0,100], [100,0], [0,-100], [-100,0], [0,-100], [-100,0], [0,100], [100,0]]
await latiss.setup_atspec(filter='RG610', grating='empty_1')
await asyncio.sleep(2)
for [xx, yy] in offsets:
    await atcs.offset_azel(az=xx, el=yy, relative=True)
    await asyncio.sleep(2)
    await latiss.take_object(2.0, 1, filter='RG610', grating='empty_1')
    await asyncio.sleep(2)
    

Applying Az/El offset: 0/0 
Timed out waiting for offset done events.
Waiting for telescope to settle.
Done
Generating group_id
imagetype: OBJECT, TCS synchronization not configured.
OBJECT 0001 - 0001
Applying Az/El offset: 0/100 
Telescope not in position.
All axes in position.
Waiting for telescope to settle.
Done
Generating group_id
imagetype: OBJECT, TCS synchronization not configured.
OBJECT 0001 - 0001
Applying Az/El offset: 100/0 
Telescope not in position.
All axes in position.
Waiting for telescope to settle.
Done
Generating group_id
imagetype: OBJECT, TCS synchronization not configured.
OBJECT 0001 - 0001
Applying Az/El offset: 0/-100 
Telescope not in position.
All axes in position.
Waiting for telescope to settle.
Done
Generating group_id
imagetype: OBJECT, TCS synchronization not configured.
OBJECT 0001 - 0001
Applying Az/El offset: -100/0 
Telescope not in position.
All axes in position.
Waiting for telescope to settle.
Done
Generating group_id
imagetype: OBJECT, TCS syn

In [ ]:
# To enable or disable built-in offsets for the filters and gratings
#await atcs.rem.ataos.cmd_disableCorrection.set_start(atspectrograph=True)
#await atcs.rem.ataos.cmd_enableCorrection.set_start(atspectrograph=True)

In [ ]:
# To stop tracking
# await atcs.stop_tracking()

In [ ]:
# If you are done, put things in standby, or shut things down completely
#await atcs.standby()
await atcs.shutdown()

In [35]:
# Putting everything back in standby.
await latiss.standby()

[atcamera]::[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]
[atspectrograph]::[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]
[atheaderservice]::[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]
[atarchiver]::[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]
All components in <State.STANDBY: 5>.
